# Limpieza de datos

La mayoría de la gente piensa que ser científico de datos significa estar corriendo modelos avanzados de Machine Learning todo el tiempo. La realidad es muy distinta:

![Tiempo de un científico de datos](./img/datascientist_time.jpeg)

La gran mayoría del tiempo se va **limpiando y organizando** los datos con los que queremos trabajar.

Los científicos de datos utilizamos herramientas como Pandas, Numpy, Matplotlib y Seaborn para limpiar los datos con los que queremos hacer algo.

## ETL

Un tipo de tarea que realizamos con gran frecuencia los científicos de datos son los **ETL**.

![Proceso de ETL](./img/etl.png)

ETL es un acrónimo que significa Extract, Transform, Load. Es un proceso que se utiliza para extraer datos de una fuente, transformarlos en un formato que sea adecuado para el análisis y cargarlos en una base de datos o algún otro sistema de almacenamiento.

## Ejemplo práctico

Los datos que usaremos para esta limpieza y nuestro siguiente análisis son datos de incidencia delictiva en nuestro país.

La iniciativa de datos abiertos del gobierno de México nos proporciona datos de incidencia delictiva desde 2015 hasta la fecha. Los datos se actualizan todos los meses y se pueden descargar desde el siguiente enlace: https://www.gob.mx/sesnsp/acciones-y-programas/datos-abiertos-de-incidencia-delictiva

---

Como podemos ver en el portal, se proporcionan los datos tanto a nivel estatal como a nivel municipal. En este caso, utilizaremos los datos a nivel estatal.

Descarguemos los datos y guardemos el archivo CSV en la carpeta data con el nombre `datos_delitos.csv`.

---

Ahora leamos el archivo CSV y veamos cómo se ven los datos.

Primero que nada, importemos pandas

In [1]:
import pandas as pd

In [2]:
# Demostración controlada: el CSV original usa ISO-8859-1, no UTF-8.
try:
    df = pd.read_csv('data/datos_delitos.csv', encoding='utf-8')
except UnicodeDecodeError:
    print('Error didáctico esperado: este archivo no se puede leer como UTF-8.')
    print('En la siguiente celda se especifica ISO-8859-1.')


Error didáctico esperado: este archivo no se puede leer como UTF-8.
En la siguiente celda se especifica ISO-8859-1.


Aquí tenemos un error muy común que suele ocurrir cuando un archivo se guarda en una computadora con cierto "encoding". 

Un encoding es una tabla que relaciona un número con un carácter. Por ejemplo, en la tabla ASCII, el número 65 corresponde a la letra "A".

Si el archivo que estamos leyendo fue guardado con un encoding distinto al que pandas espera, nos arrojará un error. 

Para solucionar esto, podemos utilizar el parámetro `encoding` de la función `pd.read_csv()` y especificar el encoding correcto.

¿Pero cómo sabemos con qué encoding cuenta el archivo?

La realidad es que la gran mayoría de los archivos los encontrarán en encoding utf-8 y Pandas no va a dar ningún error. Aquí estamos teniendo este problema porque la computadora que utilizan para generar este archivo uso un encoding diferente a utf-8.

En México, por lo general, si un archivo no está en utf-8, lo más seguro es que esté en `ISO-8859-1` o `latin1`.

Intentemos cargar el archivo especificando el encoding `ISO-8859-1`.

In [3]:
df = pd.read_csv('data/datos_delitos.csv', encoding='ISO-8859-1')
df.head()

,Año,Clave_Ent,Entidad,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
0,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,3,0,2,1,1,1,2.0,1.0,2.0,2.0,2.0,1.0
1,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0.0,1.0,0.0,0.0,0.0,1.0
2,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con otro elemento,0,0,2,2,3,2,0.0,1.0,2.0,0.0,0.0,0.0
3,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,No especificado,2,0,0,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0
4,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio culposo,Con arma de fuego,0,0,0,0,1,0,0.0,0.0,0.0,0.0,0.0,0.0


Y vemos que ya podemos leer correctamente el archivo.


¿Existe alguna forma de verificar el encoding de un archivo sin tener que estar adivinando?

ChatGTP generó el siguiente código:

In [4]:
import chardet

def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        rawdata = f.read()
    result = chardet.detect(rawdata)
    return result

file_path = './data/datos_delitos.csv'
encoding_info = detect_encoding(file_path)
print(f"Detected encoding: {encoding_info['encoding']}")

Detected encoding: ISO-8859-1


Sigamos...

In [5]:
df.tail(3)

,Año,Clave_Ent,Entidad,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
31357,2024,32,Zacatecas,Otros bienes jurídicos afectados (del fuero co...,Delitos cometidos por servidores públicos,Delitos cometidos por servidores públicos,Delitos cometidos por servidores públicos,20,22,34,35,34,37,NaN,NaN,NaN,NaN,NaN,NaN
31358,2024,32,Zacatecas,Otros bienes jurídicos afectados (del fuero co...,Electorales,Electorales,Electorales,0,1,0,11,31,23,NaN,NaN,NaN,NaN,NaN,NaN
31359,2024,32,Zacatecas,Otros bienes jurídicos afectados (del fuero co...,Otros delitos del Fuero Común,Otros delitos del Fuero Común,Otros delitos del Fuero Común,151,183,174,199,208,177,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31360 entries, 0 to 31359
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Año                     31360 non-null  int64  
 1   Clave_Ent               31360 non-null  int64  
 2   Entidad                 31360 non-null  object 
 3   Bien jurídico afectado  31360 non-null  object 
 4   Tipo de delito          31360 non-null  object 
 5   Subtipo de delito       31360 non-null  object 
 6   Modalidad               31360 non-null  object 
 7   Enero                   31360 non-null  int64  
 8   Febrero                 31360 non-null  int64  
 9   Marzo                   31360 non-null  int64  
 10  Abril                   31360 non-null  int64  
 11  Mayo                    31360 non-null  int64  
 12  Junio                   31360 non-null  int64  
 13  Julio                   28224 non-null  float64
 14  Agosto                  28224 non-null

Bien. Están muy bien los datos. Sin embargo, tenemos un problema con el que es muy común encontrarnos. 

Resulta que el formato en el que está el archivo es fácil entender por seres humanos:


```markdown
| Año      | Entidad  | Enero    | Febrero  | Mes X    |
|----------|----------|----------|----------|----------|
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
```

Vemos que tenemos los meses como encabezados. Es decir, el archivo está tratando cada mes como si fuera una variable. 

Nosotros como científicos de datos estamos más interesados en conjuntos de datos que no estén en este formato de "resumen" o "tabla dinámica". Para nosotros, lo ideal sería que cada mes fuera simplemente una observación más en nuestro conjunto de datos. Es decir, queremos transformar la tabla de arriba en:

```markdown
| Año      | Entidad  | Mes        |
|----------|----------|------------|
|   ...    |   ...    |   Enero    |
|   ...    |   ...    |   Febrero  |
|   ...    |   ...    |   Marzo    |
|   ...    |   ...    |   Abril    |
|   ...    |   ...    |   Mes X    |
```

A este tipo de formato le llamos "formato largo de datos". 

Haremos las siguientes limpiezas:
* Transformar los nombres de las columnas para que no tengan caracteres especiales y estén siempre en minúsculas
* Convertir el dataset a un formato de datos "largo"

In [7]:
for col in df.columns:
    print(col)

Año
Clave_Ent
Entidad
Bien jurídico afectado
Tipo de delito
Subtipo de delito
Modalidad
Enero
Febrero
Marzo
Abril
Mayo
Junio
Julio
Agosto
Septiembre
Octubre
Noviembre
Diciembre


In [8]:
len(df.columns)

19

### Limpiar nombres de columnas

In [9]:
def limpiar_columnas(df):
    columnas_limpias = []
    for col in df.columns:
        # convertir a minusculas, reemplazar espacios por guiones bajos y eliminar caracteres especiales
        col = col.lower().replace(" ", "_").replace("ñ", "ni").replace(".", "").replace("á", "a").replace("é", "e").replace("í","i").replace("ó", "o").replace("ú", "u")
        columnas_limpias.append(col)
    
    df.columns = columnas_limpias
    
    return df


In [10]:
df.head(2)

,Año,Clave_Ent,Entidad,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
0,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,3,0,2,1,1,1,2.0,1.0,2.0,2.0,2.0,1.0
1,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0.0,1.0,0.0,0.0,0.0,1.0


In [11]:
df = limpiar_columnas(df)

In [12]:
df.head(3)

,anio,clave_ent,entidad,bien_juridico_afectado,tipo_de_delito,subtipo_de_delito,modalidad,enero,febrero,marzo,abril,mayo,junio,julio,agosto,septiembre,octubre,noviembre,diciembre
0,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,3,0,2,1,1,1,2.0,1.0,2.0,2.0,2.0,1.0
1,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0.0,1.0,0.0,0.0,0.0,1.0
2,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con otro elemento,0,0,2,2,3,2,0.0,1.0,2.0,0.0,0.0,0.0


Muy bien. Ahora lo que queremos hacer es quitar algunas columnas. Nos interesan nada más las siguientes:

In [13]:
df[['anio', 'entidad']]

,anio,entidad
0,2015,Aguascalientes
1,2015,Aguascalientes
2,2015,Aguascalientes
3,2015,Aguascalientes
4,2015,Aguascalientes
...,...,...
31355,2024,Zacatecas
31356,2024,Zacatecas
31357,2024,Zacatecas
31358,2024,Zacatecas


In [14]:
df = df[['anio', 'clave_ent', 'entidad', 'tipo_de_delito', 'subtipo_de_delito', 'modalidad','enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio', 'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre']]
df.head(2)

,anio,clave_ent,entidad,tipo_de_delito,subtipo_de_delito,modalidad,enero,febrero,marzo,abril,mayo,junio,julio,agosto,septiembre,octubre,noviembre,diciembre
0,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con arma de fuego,3,0,2,1,1,1,2.0,1.0,2.0,2.0,2.0,1.0
1,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0.0,1.0,0.0,0.0,0.0,1.0


### Formato largo de datos

Ahora, usaremos el método `melt` para convertir las columnas a observaciones.

Queremos convervar las coumnas:
* anio
* clave_ent
* entidad
* tipo_de_delito
* subtipo_de_delito
* modalidad

El resto de las columnas las vamos a juntar en una nueva columna llamada "nombre_mes" y sus valores los vamos a sumar en otra llamada "frecuencia"

In [15]:
print("Shape ", df.shape)

Shape  (31360, 18)


In [16]:
datos_long = df.melt(id_vars=['anio', 'clave_ent', 'entidad','tipo_de_delito', 'subtipo_de_delito', 'modalidad'], var_name='nombre_mes', value_name='frecuencia')

In [17]:
print("Shape: ", datos_long.shape)

Shape:  (376320, 8)


In [18]:
datos_long.head(5)

,anio,clave_ent,entidad,tipo_de_delito,subtipo_de_delito,modalidad,nombre_mes,frecuencia
0,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con arma de fuego,enero,3.0
1,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con arma blanca,enero,1.0
2,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con otro elemento,enero,0.0
3,2015,1,Aguascalientes,Homicidio,Homicidio doloso,No especificado,enero,2.0
4,2015,1,Aguascalientes,Homicidio,Homicidio culposo,Con arma de fuego,enero,0.0


In [19]:
datos_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 376320 entries, 0 to 376319
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   anio               376320 non-null  int64  
 1   clave_ent          376320 non-null  int64  
 2   entidad            376320 non-null  object 
 3   tipo_de_delito     376320 non-null  object 
 4   subtipo_de_delito  376320 non-null  object 
 5   modalidad          376320 non-null  object 
 6   nombre_mes         376320 non-null  object 
 7   frecuencia         357504 non-null  float64
dtypes: float64(1), int64(2), object(5)
memory usage: 23.0+ MB


Supongamos que para este análisis, no nos importan los niveles subtipo de delito y modalidad. O sea, no queremos tener la distinción entre homicidios dolosos y culposos (sé que son bastante diferentes, pero simplifiquemos nuestro ejemplo).

Vamos a agrupar nuestro dataframe por anio, clave_ent, entidad, tipo_de_delito y nombre_mes. Esto hará que todos los tipos de homicidios se sumen al tipo "homicidio" o todos los tipos de robo de vehículo (con o sin violencia) se sumen a "robo de vehículo".

In [20]:
# Un mes no reportado no equivale a cero delitos.
faltantes = datos_long['frecuencia'].isna().sum()
print(f'Registros de subtipo/modalidad sin reporte que se excluyen: {faltantes:,}')
datos_long = (
    datos_long.dropna(subset=['frecuencia'])
    .groupby(['anio', 'clave_ent', 'entidad', 'tipo_de_delito', 'nombre_mes'])['frecuencia']
    .sum(min_count=1).reset_index()
)
assert (datos_long['frecuencia'] >= 0).all()
assert datos_long['frecuencia'].mod(1).eq(0).all()
datos_long['frecuencia'] = datos_long['frecuencia'].astype('int64')


Registros de subtipo/modalidad sin reporte que se excluyen: 18,816


**Criterio de limpieza y cobertura.** El CSV incluido en este repositorio contiene
enero de 2015 a junio de 2024. Sus columnas de julio a diciembre de 2024 están
vacías. Se excluyen esos meses no observados antes de agrupar: sumarlos como
ceros inventaría una caída de la incidencia. Los ceros explícitos sí se conservan.
Por eso el archivo regenerado difiere del `delitos.csv` precargado, que incluía
esos seis meses con ceros. No se modifica el archivo original `datos_delitos.csv`.

Esta agrupación conserva el **tipo** de delito, pero elimina subtipo y modalidad.
No permite distinguir homicidios dolosos de culposos ni robo de vehículo de otros
robos. En el cuaderno 3 se recuperan esos detalles desde el CSV original para
responder exactamente las preguntas. Ambos archivos son estatales, sin municipios.


In [21]:
datos_long[datos_long.tipo_de_delito == 'Robo'].sample(15, random_state=6)


,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
27275,2016,25,Sinaloa,Robo,septiembre,669
122793,2022,32,Zacatecas,Robo,noviembre,540
25353,2016,21,Puebla,Robo,noviembre,1677
48871,2018,6,Colima,Robo,marzo,637
131428,2023,18,Nayarit,Robo,febrero,118
138674,2024,2,Baja California,Robo,febrero,2902
129988,2023,15,México,Robo,febrero,10457
73834,2019,26,Sonora,Robo,octubre,763
17666,2016,5,Coahuila de Zaragoza,Robo,diciembre,1300
30624,2016,32,Zacatecas,Robo,abril,618


Mostremos todos los estados y su respectiva clave

In [22]:
datos_long[['clave_ent', 'entidad']].drop_duplicates().sort_values('entidad')

,clave_ent,entidad
0,1,Aguascalientes
480,2,Baja California
960,3,Baja California Sur
1440,4,Campeche
2880,7,Chiapas
3360,8,Chihuahua
3840,9,Ciudad de México
1920,5,Coahuila de Zaragoza
2400,6,Colima
4320,10,Durango


Ahora Veamos todos los datos de delitos de una entidad en específico. Por ejemplo, Nuevo león

In [23]:
datos_long[datos_long['clave_ent'] == 19].sample(15, random_state=6)


,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
142733,2024,19,Nuevo León,Otros delitos que atentan contra la libertad p...,mayo,152
85829,2020,19,Nuevo León,Robo,julio,1153
101057,2021,19,Nuevo León,Incumplimiento de obligaciones de asistencia f...,julio,44
131678,2023,19,Nuevo León,Extorsión,diciembre,64
116439,2022,19,Nuevo León,Narcomenudeo,enero,307
39424,2017,19,Nuevo León,Amenazas,febrero,369
24324,2016,19,Nuevo León,Otros delitos del Fuero Común,abril,224
70233,2019,19,Nuevo León,Evasión de presos,noviembre,1
24004,2016,19,Nuevo León,Aborto,febrero,2
85794,2020,19,Nuevo León,Otros delitos que atentan contra la libertad y...,junio,64


In [24]:
datos_long[(datos_long['clave_ent'] > 19) &  (datos_long['clave_ent'] < 24) & (datos_long['tipo_de_delito'] == 'Homicidio')].sample(15, random_state=6)


,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
55902,2018,21,Puebla,Homicidio,junio,163
102459,2021,22,Querétaro,Homicidio,enero,39
10784,2015,23,Quintana Roo,Homicidio,mayo,44
26147,2016,23,Quintana Roo,Homicidio,septiembre,26
116866,2022,20,Oaxaca,Homicidio,octubre,183
116860,2022,20,Oaxaca,Homicidio,febrero,113
26145,2016,23,Quintana Roo,Homicidio,noviembre,30
25185,2016,21,Puebla,Homicidio,noviembre,65
56386,2018,22,Querétaro,Homicidio,octubre,25
71737,2019,22,Querétaro,Homicidio,agosto,43


### Valores de fechas

Finalmente, queremos tener una columna "fecha". Actualmente tenemos el año y el nombre del mes, pero no tenemos como tal una columna que tenga un tipo de dato fecha. Eso hace que filtrar por fecha sea complicado.

Por ejemplo, si queremos conocer todos los homicidios de Oaxaca en enero 2024, haríamos lo siguiente:

In [25]:
datos_long[
    (datos_long['clave_ent'] == 20) &
    (datos_long['tipo_de_delito'] == 'Homicidio') &
    (datos_long['anio'] == 2024) &
    (datos_long['nombre_mes'] == 'enero') 
]

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
142909,2024,20,Oaxaca,Homicidio,enero,157


Creemos una columna de fecha. 

Primero tenemos que convertir el nombre de mes a un número, en donde 1 es enero, 2 febrero, etc.

In [26]:
datos_long.sample(2, random_state=6)


,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
1122,2015,3,Baja California Sur,Extorsión,junio,5
125234,2023,5,Coahuila de Zaragoza,Violación equiparada,diciembre,37


In [27]:
datos_long["nueva_columna"] = "dato vacío"
datos_long.sample(6, random_state=6)


,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna
1122,2015,3,Baja California Sur,Extorsión,junio,5,dato vacío
125234,2023,5,Coahuila de Zaragoza,Violación equiparada,diciembre,37,dato vacío
21025,2016,12,Guerrero,Robo,agosto,1053,dato vacío
12433,2015,26,Sonora,Violación equiparada,agosto,0,dato vacío
61616,2019,1,Aguascalientes,Falsedad,mayo,8,dato vacío
141750,2024,15,México,Otros delitos contra la familia,abril,1,dato vacío


In [28]:
# Diccionario de ayuda para convertir
meses = {
    "enero": 1,
    "febrero": 2,
    "marzo": 3,
    "abril": 4,
    "mayo": 5,
    "junio": 6,
    "julio": 7,
    "agosto": 8,
    "septiembre": 9,
    "octubre": 10,
    "noviembre": 11,
    "diciembre": 12
}

datos_long['mes'] = datos_long['nombre_mes'].map(meses)
datos_long.sample(5, random_state=6)


,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna,mes
1122,2015,3,Baja California Sur,Extorsión,junio,5,dato vacío,6
125234,2023,5,Coahuila de Zaragoza,Violación equiparada,diciembre,37,dato vacío,12
21025,2016,12,Guerrero,Robo,agosto,1053,dato vacío,8
12433,2015,26,Sonora,Violación equiparada,agosto,0,dato vacío,8
61616,2019,1,Aguascalientes,Falsedad,mayo,8,dato vacío,5


In [29]:
datos_long["frecuencia_mas_10"] = datos_long["frecuencia"] + 10
datos_long.sample(5, random_state=6)


,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna,mes,frecuencia_mas_10
1122,2015,3,Baja California Sur,Extorsión,junio,5,dato vacío,6,15
125234,2023,5,Coahuila de Zaragoza,Violación equiparada,diciembre,37,dato vacío,12,47
21025,2016,12,Guerrero,Robo,agosto,1053,dato vacío,8,1063
12433,2015,26,Sonora,Violación equiparada,agosto,0,dato vacío,8,10
61616,2019,1,Aguascalientes,Falsedad,mayo,8,dato vacío,5,18


In [30]:
datos_long["anio_mes"] = datos_long["anio"].astype(str) + datos_long["mes"].astype(str)
datos_long.sample(6, random_state=6)


,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna,mes,frecuencia_mas_10,anio_mes
1122,2015,3,Baja California Sur,Extorsión,junio,5,dato vacío,6,15,20156
125234,2023,5,Coahuila de Zaragoza,Violación equiparada,diciembre,37,dato vacío,12,47,202312
21025,2016,12,Guerrero,Robo,agosto,1053,dato vacío,8,1063,20168
12433,2015,26,Sonora,Violación equiparada,agosto,0,dato vacío,8,10,20158
61616,2019,1,Aguascalientes,Falsedad,mayo,8,dato vacío,5,18,20195
141750,2024,15,México,Otros delitos contra la familia,abril,1,dato vacío,4,11,20244


In [31]:
# yyyy-mm-dd, yy-mm-dd, yymmdd, yyyy/dd/mm

In [32]:
# Agregamos la columna de fecha juntando el año y el mes
datos_long['fecha'] = pd.to_datetime(datos_long['anio'].astype(str) + datos_long['mes'].astype(str), format='%Y%m')
datos_long.sample(5, random_state=6)


,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna,mes,frecuencia_mas_10,anio_mes,fecha
1122,2015,3,Baja California Sur,Extorsión,junio,5,dato vacío,6,15,20156,2015-06-01
125234,2023,5,Coahuila de Zaragoza,Violación equiparada,diciembre,37,dato vacío,12,47,202312,2023-12-01
21025,2016,12,Guerrero,Robo,agosto,1053,dato vacío,8,1063,20168,2016-08-01
12433,2015,26,Sonora,Violación equiparada,agosto,0,dato vacío,8,10,20158,2015-08-01
61616,2019,1,Aguascalientes,Falsedad,mayo,8,dato vacío,5,18,20195,2019-05-01


In [33]:
# Eliminamos las columnas que ya no necesitamos
datos_long = datos_long.drop(columns=['nueva_columna'])
datos_long.sample(5, random_state=6)


,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,mes,frecuencia_mas_10,anio_mes,fecha
1122,2015,3,Baja California Sur,Extorsión,junio,5,6,15,20156,2015-06-01
125234,2023,5,Coahuila de Zaragoza,Violación equiparada,diciembre,37,12,47,202312,2023-12-01
21025,2016,12,Guerrero,Robo,agosto,1053,8,1063,20168,2016-08-01
12433,2015,26,Sonora,Violación equiparada,agosto,0,8,10,20158,2015-08-01
61616,2019,1,Aguascalientes,Falsedad,mayo,8,5,18,20195,2019-05-01


Veamos los homicidios en oaxaca de enero 2024 a la fecha

In [34]:
datos_long[
    (datos_long.tipo_de_delito == "Homicidio") &
    (datos_long.clave_ent == 20) &
    (datos_long.fecha >= '2024-01-01')
].sort_values('fecha')

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,mes,frecuencia_mas_10,anio_mes,fecha
142909,2024,20,Oaxaca,Homicidio,enero,157,1,167,20241,2024-01-01
142910,2024,20,Oaxaca,Homicidio,febrero,170,2,180,20242,2024-02-01
142912,2024,20,Oaxaca,Homicidio,marzo,190,3,200,20243,2024-03-01
142908,2024,20,Oaxaca,Homicidio,abril,208,4,218,20244,2024-04-01
142913,2024,20,Oaxaca,Homicidio,mayo,236,5,246,20245,2024-05-01
142911,2024,20,Oaxaca,Homicidio,junio,202,6,212,20246,2024-06-01


In [35]:
datos_finales = datos_long[
    ['anio', 'clave_ent', 'entidad', 'tipo_de_delito', 'nombre_mes', 'fecha', 'frecuencia']
].sort_values(['fecha', 'clave_ent', 'tipo_de_delito']).reset_index(drop=True).copy()

assert pd.api.types.is_datetime64_any_dtype(datos_finales['fecha'])
assert datos_finales['fecha'].dt.year.eq(datos_finales['anio']).all()
assert datos_finales['fecha'].dt.day.eq(1).all()
assert not datos_finales.duplicated(['fecha', 'clave_ent', 'tipo_de_delito']).any()
assert datos_finales['clave_ent'].nunique() == 32
print(f'Filas finales: {len(datos_finales):,}; entidades: 32')
print(f"Cobertura observada: {datos_finales.fecha.min():%Y-%m} a {datos_finales.fecha.max():%Y-%m}")
print(f"Frecuencia total conservada: {datos_finales.frecuencia.sum():,}")
display(datos_finales.groupby('anio')['fecha'].nunique().rename('meses_observados'))
display(datos_finales.head(2))


Filas finales: 145,920; entidades: 32
Cobertura observada: 2015-01 a 2024-06
Frecuencia total conservada: 18,694,886


anio
2015    12
2016    12
2017    12
2018    12
2019    12
2020    12
2021    12
2022    12
2023    12
2024     6
Name: meses_observados, dtype: int64

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,fecha,frecuencia
0,2015,1,Aguascalientes,Aborto,enero,2015-01-01,0
1,2015,1,Aguascalientes,Abuso de confianza,enero,2015-01-01,41


Ya que tenemos muestros datos bien estructurados, los podemos guardar en nuestra computadora. Los guardaremos con el nombre "delitos.csv"

In [36]:
datos_finales.to_csv('data/delitos.csv', index=False, encoding='utf-8')
verificacion = pd.read_csv('data/delitos.csv', parse_dates=['fecha'])
pd.testing.assert_frame_equal(datos_finales, verificacion)
print('data/delitos.csv exportado y verificado mediante una nueva lectura.')


data/delitos.csv exportado y verificado mediante una nueva lectura.
